# 2 · Find and validate

For one year (built with notebook 1): apply the current rules, rank the papers by similarity to
the confirmed facility papers, and **review them here**, most likely first: for each paper a link
to the published version and to an open copy, and buttons **yes / likely / no / skip**. Every
click is saved in your Drive at once: stop whenever you like, and run the review cell again to
continue where you left off.

Then section 5 writes the **corresponding authors** of the facility's papers to a CSV file in
your Drive (for an e-mail blast), opens an *Add papers* issue with the confirmed DOIs (an agent
files them in a pull request you merge), and re-ranks the other years with what you decided.

All files stay in your Drive folder (`DATA`), under the year:

| file | what |
|---|---|
| `validate.csv` | the papers to review, with your verdict, reason and note |
| `contacts.csv` | corresponding authors of the confirmed and validated papers: one row per e-mail address |
| `search_misses.csv` | confirmed papers the search did not flag, and why (for whoever maintains the rules) |

## Settings

In [ ]:
FACILITY = 'aic-turku'
YEAR = 2025                               # the year to validate (built with notebook 1)
TOP_N = 200                               # check-list papers to include ("reads like facility papers")
AT_ONCE = 1                               # papers shown at a time in the review
DATA = '/content/drive/MyDrive/AIC-publications'
REPO = 'https://github.com/AIC-Turku/facility-publication-finder'

## 1. Setup

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA, exist_ok=True)

# The code is cloned fresh into the temporary Colab disk every session; all data stays in DATA
# (Google Drive), so updating the code never touches results.
CODE = '/content/code'                    # temporary disk; rules and confirmed papers come with it
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '--ff-only', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, CODE], check=True)
%pip install -q "{CODE}[embed,openiris]"
os.environ['PUBS_DATA'] = DATA                                  # every command reads and writes data here
os.environ['PUBS_FACILITY'] = f'{CODE}/facilities/{FACILITY}'   # rules + confirmed papers
!facility-pubs check-facility

from facility_pubs import review, workflow

## 2. Search the year with the current rules

In [ ]:
if YEAR not in workflow.swept_years():
    raise SystemExit(f'{YEAR} is not built yet: run notebook 1 for it first.')
!facility-pubs rescreen --year {YEAR}
!facility-pubs validate --years {YEAR}

## 3. Rank

Lists the papers to review, most likely first: those that acknowledge the facility, then those
naming a facility instrument, then the top `TOP_N` of the check list (papers that read like the
facility's papers). The ranking learns from the confirmed papers in the repository and from
your decisions in other years (yes as examples, no as counter-examples), never from this
year's own papers. Decisions already made are kept.

In [ ]:
for path in workflow.prepare(YEAR, top_n=TOP_N):
    print('wrote', path)

## 4. Review

Open the links (each opens in a new tab), read the methods and acknowledgements, come back and
decide:

* **yes**: the paper used the facility. It will be filed, and its authors are in `contacts.csv`.
* **likely**: probably, but not certain. Not filed; its authors are in `contacts.csv` (status
  *likely*) so you can ask them; review it again later (edit `validate.csv`, or set its verdict
  to empty to see it here again).
* **no**: not facility use.
* **skip**: decide later (it comes back next time).

A **reason** (optional, pick it before clicking) and a **note** help improve the rules and the
ranking (section 6). Run this cell again any time to continue.

In [ ]:
review.start(YEAR, at_once=AT_ONCE)

## 5. Contacts and filing

Refreshes the tables: `contacts.csv` lists the corresponding authors of every confirmed paper
of the year and of those you marked yes or likely, one row per e-mail address (an author of
several papers appears once, with all their papers). Open it in Drive (or download it) for the
e-mail blast. It is rewritten on every refresh: edit a copy.

The addresses were not given by the authors: the first message should say where the address
comes from, why it is used, and how to opt out; keep a do-not-contact list, and delete the file
after the campaign. Mail-merge add-ons are third-party processors (check with your data
protection officer).

Then file the confirmed papers: open the printed link (an **Add papers** issue with the DOIs
filled in), submit it and comment `@claude file these`. The agent opens a pull request that files
them under `papers/<year>.yaml` (marked as reviewed by the staff); its check confirms every DOI
resolves; merge it. Re-run *1. Setup* in a later session to pick up the merged papers.

In [ ]:
contacts, to_file = workflow.collect(YEAR, top_n=TOP_N)
print('Corresponding authors:', contacts)
if to_file:
    print('\nOpen the Add papers issue (DOIs filled in), submit it, comment "@claude file these":')
    print(workflow.issue_link(REPO, FACILITY, to_file))
    print('\n' + to_file)
else:
    print('(no new "yes" verdicts to file)')

## 6. Learn and improve

Re-ranks the other years with your decisions, then summarises what the decisions say: how often
each kind of candidate was a facility paper, how far down the check list facility papers were
still found (to choose `TOP_N`), why papers were not facility use (which rule to fix first), and
the facility papers only the check list found (patterns to add). Saved as `feedback.txt` in
`DATA`; send it to whoever maintains the rules.

In [ ]:
for y in workflow.learn(YEAR, top_n=TOP_N):
    print('re-ranked', y)
report, path = workflow.feedback()
print(report)
print('\nsaved to', path)